# Yandu Wang WIP

Calculate three numbers

1. **canonical order count**
2. **one file within-file duplicate count** order_id
3. **teo file  cross-file overlap count**  order_id

# 0. path

- **Colab** and **VS Code **

In [ ]:
from pathlib import Path

GROUP_ID = "001"

# Colab  Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# VS Code
_colab_base = Path("/content/drive/MyDrive/Group001_A1")
BASE = _colab_base if _colab_base.is_dir() else Path(".")

INPUT_DIR = BASE / "DATA" / f"Group{GROUP_ID}_A1" / "raw_input"
JSON_NAME = f"Group{GROUP_ID}_commerce.json"
XML_NAME  = f"Group{GROUP_ID}_operations.xml"

def _resolve(name):

    primary = INPUT_DIR / name
    if primary.exists():
        return primary
    for c in [Path(name), Path("raw_input") / name]:
        if c.exists():
            return c
    matches = list(BASE.rglob(name))
    if matches:
        return matches[0]
    raise FileNotFoundError(
        f"no {name}. pl sure in DATA/Group{GROUP_ID}_A1/raw_input/"
    )

JSON_PATH = _resolve(JSON_NAME)
XML_PATH  = _resolve(XML_NAME)
print("JSON:", JSON_PATH)
print("XML :", XML_PATH)


Mounted at /content/drive
JSON: /content/drive/MyDrive/Group001_A1/DATA/Group001_A1/raw_input/Group001_commerce.json
XML : /content/drive/MyDrive/Group001_A1/DATA/Group001_A1/raw_input/Group001_operations.xml


## 1. read two file

ues json and xml.etree.ElementTree

In [ ]:
import json
import xml.etree.ElementTree as ET

# ---- JSON (commerce out) ----
with open(JSON_PATH, encoding="utf-8") as f:
    jdata = json.load(f)

print("JSON top:", list(jdata.keys()))
print("orders list lenth:", len(jdata["orders"]))

# ---- XML (operations ERP out) ----
xroot = ET.parse(XML_PATH).getroot()
n_order_nodes = len(list(xroot.iter("Order")))
print("XML <Order> point number:", n_order_nodes)


JSON top: ['customerProfiles', 'exportMetadata', 'orders', 'productReviews']
orders list lenth: 2818
XML <Order> point number: 2818


## 2. get order_id
# Count orders from Header/Order_ID only — it also repeats in items and delivery.

In [ ]:
# Values ​​(retaining duplicates)
json_ids = [o["header"]["orderID"] for o in jdata["orders"]]
xml_ids = []
for order in xroot.iter("Order"):
    header = order.find("Header")
    if header is not None:
        oid = header.find("Order_ID")
        if oid is not None and oid.text:
            xml_ids.append(oid.text.strip())

print("JSON Number of order records (Contains duplicates):", len(json_ids))
print("XML  Number of order records (Contains duplicates):", len(xml_ids))


JSON Number of order records (Contains duplicates): 2818
XML  Number of order records (Contains duplicates): 2818


## 3. calculate  within-file duplicate count

In [ ]:
from collections import Counter

jc = Counter(json_ids)
xc = Counter(xml_ids)

json_within_dups = sum(1 for _id, n in jc.items() if n > 1)
xml_within_dups  = sum(1 for _id, n in xc.items() if n > 1)

print("JSON Number of duplicate order_ids in the file:", json_within_dups)
print("XML  Number of duplicate order_ids in the file:", xml_within_dups)

WITHIN_FILE_DUPLICATE_COUNT = json_within_dups
print("within-file duplicate count =", WITHIN_FILE_DUPLICATE_COUNT)


JSON Number of duplicate order_ids in the file: 68
XML  Number of duplicate order_ids in the file: 68
within-file duplicate count = 68


## 4. calculate cross-file overlap count

In [ ]:
json_unique = set(json_ids)
xml_unique  = set(xml_ids)

overlap_ids = json_unique & xml_unique
CROSS_FILE_OVERLAP_COUNT = len(overlap_ids)

print("JSON Different after deduplication order_id:", len(json_unique))
print("XML  Different after deduplication order_id:", len(xml_unique))
print("cross-file overlap count =", CROSS_FILE_OVERLAP_COUNT)


JSON Different after deduplication order_id: 2750
XML  Different after deduplication order_id: 2750
cross-file overlap count = 500


## 5. calculate canonical order count

Remove duplicates, then merge the two files; what remains is the actual number of orders.

In [ ]:
canonical_ids = json_unique | xml_unique
CANONICAL_ORDER_COUNT = len(canonical_ids)

# all = J + X - dp
check = len(json_unique) + len(xml_unique) - CROSS_FILE_OVERLAP_COUNT
assert check == CANONICAL_ORDER_COUNT, (check, CANONICAL_ORDER_COUNT)

print(">>> canonical order count =", CANONICAL_ORDER_COUNT)


>>> canonical order count = 5000


## 6. three numbers

In [ ]:

print(f"  1. canonical order count       = {CANONICAL_ORDER_COUNT}")
print(f"  2. within-file duplicate count = {WITHIN_FILE_DUPLICATE_COUNT}  (each file: {WITHIN_FILE_DUPLICATE_COUNT})")
print(f"  3. cross-file overlap count    = {CROSS_FILE_OVERLAP_COUNT}")
print(f"{len(json_unique)} + {len(xml_unique)} - {CROSS_FILE_OVERLAP_COUNT} = {CANONICAL_ORDER_COUNT}")

  1. canonical order count       = 5000
  2. within-file duplicate count = 68  (each file: 68)
  3. cross-file overlap count    = 500
2750 + 2750 - 500 = 5000
